# Detection criteria and lightcurve quality analysis

This notebook analyzes outputs produced by `scripts/run_detection_criteria_sweep.sh`. It does not run simulations.

In [ ]:
from pathlib import Path
import json
import re
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path('results/detection_criteria_sweep')
QUALITY = ROOT / 'lightcurve_quality.csv'
if not QUALITY.exists():
    raise FileNotFoundError(f'Run lightcurve_quality_analysis.py first: {QUALITY}')
quality = pd.read_csv(QUALITY)
quality.head()

## Lightcurve quality

The configured span is the detector's qualifying window. The all-observation span is reported separately because stored non-detections may extend beyond that window.

In [ ]:
numeric = [
    'n_positive', 'n_non_detections', 'n_bands', 'duplicate_positive_times',
    'positive_span_days', 'all_observation_span_days', 'max_positive_gap_days',
    'min_positive_gap_days', 'configured_max_timespan_days'
]
quality[numeric] = quality[numeric].apply(pd.to_numeric, errors='coerce')
quality['positive_span_exceeds_configured_days'] = quality['positive_span_exceeds_configured_days'].astype(bool)
quality['all_observation_span_exceeds_configured_days'] = quality['all_observation_span_exceeds_configured_days'].astype(bool)
quality[numeric].describe(percentiles=[0.5, 0.9, 0.95, 0.99]).T

In [ ]:
quality_summary = (
    quality.groupby([
        'criteria_snr_threshold', 'criteria_snr_threshold_secondary',
        'criteria_min_detections', 'criteria_min_detections_primary',
        'configured_max_timespan_days', 'criteria_min_time_separation_hours',
        'criteria_min_fade_rate', 'criteria_min_galactic_lat',
    ], dropna=False)
    .agg(
        sources=('source_index', 'size'),
        median_positive_span_days=('positive_span_days', 'median'),
        p95_positive_span_days=('positive_span_days', lambda values: values.quantile(0.95)),
        median_max_gap_days=('max_positive_gap_days', 'median'),
        positive_span_violations=('positive_span_exceeds_configured_days', 'sum'),
        all_span_violations=('all_observation_span_exceeds_configured_days', 'sum'),
        duplicate_time_sources=('duplicate_positive_times', lambda values: (values > 0).sum()),
    )
    .reset_index()
)
quality_summary.sort_values('positive_span_violations', ascending=False).head(20)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 4.5))
quality['positive_span_days'].hist(ax=axes[0], bins=50)
axes[0].set(xlabel='positive-detection span [days]', ylabel='detected sources')
quality['max_positive_gap_days'].hist(ax=axes[1], bins=50)
axes[1].set(xlabel='largest positive-detection gap [days]', ylabel='detected sources')
quality['n_bands'].value_counts().sort_index().plot.bar(ax=axes[2])
axes[2].set(xlabel='number of observed bands', ylabel='detected sources')
fig.tight_layout()

## Rate-limit comparison

Each simulation result is paired with its metadata sidecar. The rate table below is built directly from the serialized `rate_summaries`; no rates are recomputed from the lightcurve audit.

In [ ]:
rate_rows = []
for result_path in sorted(ROOT.glob('**/*.json')):
    if result_path.name.endswith('.metadata.json'):
        continue
    with result_path.open() as handle:
        result = json.load(handle)
    metadata_path = Path(f'{result_path}.metadata.json')
    metadata = json.loads(metadata_path.read_text()) if metadata_path.exists() else {}
    criteria = metadata.get('detection_criteria', {})
    for summary in result.get('rate_summaries', []):
        row = dict(summary)
        row.update({
            'file': str(result_path),
            'snr_threshold': criteria.get('snr_threshold'),
            'snr_threshold_secondary': criteria.get('snr_threshold_secondary'),
            'min_detections': criteria.get('min_detections'),
            'min_detections_primary': criteria.get('min_detections_primary'),
            'max_timespan_days': criteria.get('max_timespan_days'),
            'min_time_separation_hours': criteria.get('min_time_separation_hours'),
            'min_fade_rate': criteria.get('min_fade_rate'),
            'min_galactic_lat': criteria.get('min_galactic_lat'),
        })
        rate_rows.append(row)
rates = pd.DataFrame(rate_rows)
if rates.empty:
    raise ValueError(f'No rate summaries found under {ROOT}')
rates['rate_95_zero_events'] = -__import__('numpy').log(0.05) / pd.to_numeric(rates['effective_vt_gpc3_yr'])
rates.sort_values(['snr_threshold', 'snr_threshold_secondary', 'min_detections', 'max_timespan_days']).head(20)

In [ ]:
rate_summary = (
    rates.groupby([
        'snr_threshold', 'snr_threshold_secondary', 'min_detections',
        'min_detections_primary', 'max_timespan_days',
        'min_time_separation_hours', 'min_fade_rate', 'min_galactic_lat',
    ], dropna=False)
    .agg(
        runs=('file', 'nunique'),
        detected=('n_detected', 'mean'),
        efficiency=('overall_efficiency', 'mean'),
        vt_eff=('effective_vt_gpc3_yr', 'mean'),
        rate_95=('rate_95_zero_events', 'mean'),
    )
    .reset_index()
)
rate_summary.sort_values('rate_95')

In [ ]:
plot_rates = rate_summary[
    (rate_summary['min_detections_primary'] == 1)
    & (rate_summary['min_time_separation_hours'] == 3)
    & (rate_summary['min_fade_rate'] == 0.3)
    & (rate_summary['min_galactic_lat'] == 15)
]
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
for (snr, secondary, minimum), subset in plot_rates.groupby(
    ['snr_threshold', 'snr_threshold_secondary', 'min_detections']
):
    subset = subset.sort_values('max_timespan_days')
    label = f'SNR={snr:g}, secondary={secondary:g}, detections={int(minimum)}'
    axes[0].plot(subset['max_timespan_days'], subset['rate_95'], marker='o', label=label)
axes[0].set(xlabel='maximum qualifying timespan [days]', ylabel='95% zero-event rate limit')
axes[0].legend(fontsize=8, ncol=2)
pivot = plot_rates.pivot_table(
    index='max_timespan_days', columns='snr_threshold', values='efficiency', aggfunc='mean'
)
pivot.plot(marker='o', ax=axes[1])
axes[1].set(xlabel='maximum qualifying timespan [days]', ylabel='mean detection efficiency')
fig.tight_layout()